![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 8 -- Lab 1: Price a Used Car

A used-car website in Saudi Arabia wants a tool that suggests a fair price for a car from its listing: make, model year, engine size, mileage, options, gear type and origin. Today you build that tool with a linear regression, and you measure, step by step, how much each piece of data work (cleaning, encoding the text columns, scaling, transforming the target) improves the model. The modelling is two lines; the improvement comes from what you feed it.

**Your role:** Fill in the `# Your code here` cells in order. Run every cell, including the ones already written for you.

**Dataset:** Real listings scraped from syarah.com, Kaggle `turkibintalib/saudi-arabia-used-cars-dataset` (8,035 rows, 13 columns). The setup cell downloads it with `kagglehub` and loads `UsedCarsSA_Clean_EN.csv` into a DataFrame `df`.

---
# Setup

In [ ]:
# kagglehub, pandas, matplotlib and scikit-learn are preinstalled on Colab. Uncomment if an import fails.
# !pip install -q kagglehub pandas matplotlib scikit-learn

In [ ]:
import os
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
path = kagglehub.dataset_download("turkibintalib/saudi-arabia-used-cars-dataset")
df = pd.read_csv(os.path.join(path, "UsedCarsSA_Clean_EN.csv"))
print(df.shape)

In [ ]:
def report(y_true, y_pred):
    """The three regression scores of the Metrics slide, printed on one line."""
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    r2 = r2_score(y_true, y_pred)
    print(f"MAE {mae:,.0f}   RMSE {rmse:,.0f}   R2 {r2:.3f}")
    return mae, rmse, r2

---
# Part 1 -- Look before you model

The *ML Tasks* slides: find the target, find which columns are numbers and which are text, and look for traps. Every fix you make here will be measured in Part 3.

## Task 1: First rows and types

Display `df.head()` and then `df.dtypes`. Which column is the target? Which columns are text (`object`)? Write the answers in a comment in the cell.

In [ ]:
# Your code here


## Task 2: Count the categories

For each of `Options`, `Gear_Type` and `Origin`, print `df[col].value_counts()`. One of the three has a natural order (worst to best). Which one?

In [ ]:
# Your code here


## Task 3: A trap in the target

Display `df["Price"].describe()`. The minimum is 0: a price of zero is not a price. Count the rows with `Price == 0` and store the count in `n_zero` (expected 2527). Then look at `df["Negotiable"].sum()`: what do the zero-price rows mean?

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert n_zero == 2527, 'n_zero: count the rows where df["Price"] == 0'
print('Task 3 passed')

## Task 4: Keep only real prices, drop duplicates

Replace `df` by the rows with `Price > 0`, then by `df.drop_duplicates().reset_index(drop=True)`. Print the shape. Expected: `(5506, 13)`.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert df.shape == (5506, 13), 'df: keep Price > 0, then drop_duplicates().reset_index(drop=True)'
print('Task 4 passed')

## Task 5: Does the ordered column matter?

Print the median price per `Options` group with `groupby`. Expected: Standard 45,000, Semi Full 55,000, Full 81,500. The order of the categories is the order of the prices: that is why an ordered code (0, 1, 2) will make sense in Part 3.

In [ ]:
# Your code here


## Task 6: Year against price

Draw a scatter plot of `Year` (x) against `Price` (y) with a title and axis labels. Two kinds of outliers are visible: a few cars from before 2000 and a few prices above 500,000. Keep them for now; Part 3 measures what removing them is worth.

In [ ]:
# Your code here


---
# Part 2 -- The first model

The *scikit-learn* slide: `fit` on the training rows, `predict` on rows the model has not seen, score with the *Metrics* slide's numbers. `train_test_split` keeps 20 percent of the rows aside; `random_state=42` makes everyone's split identical.

## Task 7: Split, fit, score on numbers only

Build `X = df[["Year", "Engine_Size", "Mileage"]]` and `y = df["Price"]`. Split with `train_test_split(X, y, test_size=0.2, random_state=42)` into `X_train, X_val, y_train, y_val`. Fit a `LinearRegression` on the training part, predict `X_val`, and call `report(y_val, pred)`. Expected: MAE about 40,000 and R2 about 0.22. Write the three scores in the comment: this is the line to beat.

Syntax hint (the shape of the call, not the answer):

```python
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
```

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert abs(scores[0] - 40121) < 300, 'MAE should be about 40,100: three numeric columns, split with random_state=42, fit on X_train only'
print('Task 7 passed')

## Task 8: A function that does the whole thing

You will repeat split, fit, predict, report six more times. Write `fit_and_score(X, y)` that does exactly Task 7 and returns the fitted model. Call it on the same `X` and `y`: it must print the same three scores.

In [ ]:
# Your code here


---
# Part 3 -- Feed the model better data

Each task changes one thing and re-runs `fit_and_score`. Keep a note of the MAE after every task: that list is the lesson.

## Task 9: Label encoding: the ordered column and the two-value column

A linear model needs numbers. `Options` has an order, so map it to codes that keep the order: `{"Standard": 0, "Semi Full": 1, "Full": 2}` into a new column `Options_code`. Map `Gear_Type` to `{"Manual": 0, "Automatic": 1}` into `Gear_code`. Then set `X = df[["Year", "Engine_Size", "Mileage", "Options_code", "Gear_code"]]` and run `fit_and_score(X, y)`. Expected MAE about 38,100.

Syntax hint (the shape of the call, not the answer):

```python
df["new_col"] = df["text_col"].map({"value A": 0, "value B": 1})
```

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert df['Options_code'].isna().sum() == 0 and set(df['Options_code'].unique()) == {0, 1, 2}, 'Options_code: map the three exact category names to 0, 1, 2'
assert set(df['Gear_code'].unique()) == {0, 1}, 'Gear_code: Manual -> 0, Automatic -> 1'
print('Task 9 passed')

## Task 10: One-hot encoding: the unordered columns

`Origin` (Saudi, Gulf Arabic, Other, Unknown) and `Fuel_Type` have no order, so a code 0, 1, 2, 3 would invent one. Use one-hot encoding instead: `onehot = pd.get_dummies(df[["Origin", "Fuel_Type"]], dtype=int)` gives one 0/1 column per category (7 columns). Print `onehot.head(3)`, then build `X = pd.concat([df[numeric], onehot], axis=1)` (12 columns) and run `fit_and_score`. Expected MAE about 37,600.

Syntax hint (the shape of the call, not the answer):

```python
onehot = pd.get_dummies(df[[col1, col2]], dtype=int)
X = pd.concat([left_table, right_table], axis=1)
```

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert X.shape[1] == 12, 'X: the 5 numeric columns plus the 7 one-hot columns of Origin and Fuel_Type'
print('Task 10 passed')

## Task 11: One-hot encoding of the make

The car's `Make` (Toyota, Hyundai, ...) has 59 values and no order. One-hot encode it with `pd.get_dummies(df["Make"], prefix="Make", dtype=int)`, add the columns to `X` with `pd.concat` (69 columns now: 57 makes remain after Task 4) and run `fit_and_score`. Expected MAE about 29,300: the biggest single gain of the lab.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert X.shape[1] == 69, 'X: the 12 columns of Task 10 plus one 0/1 column per make (57 makes remain after Task 4)'
print('Task 11 passed')

## Task 12: Remove the outliers you saw in the scatter plot

Build a boolean mask `keep` for rows with `Year >= 2000`, `Mileage <= 1_000_000` and `Price >= 5000` (Day 05: combine masks with `&`). Print `keep.sum()` (expected 5285), then run `fit_and_score(X[keep], y[keep])`. Expected MAE about 27,000. Keep `X_clean = X[keep]` and `y_clean = y[keep]` for the next tasks.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert keep.sum() == 5285, 'keep: three conditions joined with &, each in parentheses'
print('Task 12 passed')

## Task 13: Scaling: same scores, readable weights

Fit `StandardScaler()` on `X_clean[["Year", "Engine_Size", "Mileage"]]`, transform them into `X_scaled`, and fit two `LinearRegression` models on the whole `y_clean`: one on the raw three columns, one on `X_scaled`. Print both `coef_` arrays. The raw weights are in mixed units (SAR per year, per litre, per kilometre) and cannot be compared; the scaled weights answer 'which feature matters most?'. Scaling does not change a linear regression's predictions, but Lab 3's logistic regression will need it.

Syntax hint (the shape of the call, not the answer):

```python
scaler = StandardScaler().fit(table)
scaled_array = scaler.transform(table)
```

In [ ]:
# Your code here


## Task 14: Transform the target: predict log price

Prices are skewed (many cars near 50,000, a few near 1,000,000), and squared errors let the expensive cars dominate. Fit on `np.log(y_train)` instead, and turn predictions back with `np.exp`. Write the split-fit-predict by hand this time (the same four lines as Task 7 on `X_clean, y_clean`), with `np.log` around `y_train` in `fit` and `np.exp` around the prediction, then `report`. Expected MAE about 20,200 and R2 about 0.75.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert final[2] > 0.7, 'R2 should be about 0.75: fit on np.log(y_train), predict, then np.exp before report'
print('Task 14 passed')

## Task 15: Price one car

Take the first row of `X_val` with `X_val.iloc[[0]]` (double brackets keep it a table), predict its price with the last model (remember `np.exp`), and print it next to the true price `y_val.iloc[0]`.

In [ ]:
# Your code here


---
## Task 16: Reflect

Write the MAE after each of Tasks 7, 9, 10, 11, 12 and 14 as a list. Which single step helped most, and why do you think a linear model needed it? Why would label encoding (0, 1, 2, 3) have been wrong for `Origin` but right for `Options`? The site's target is a price the seller will accept: is MAE or RMSE the better score to report to them, and why?

*Your answers here*

---
## Conclusion

- **Look before you model**: you found the target, the text columns, the zero prices that meant 'negotiable', a duplicated row and the outliers, all before fitting anything.
- **Linear regression in scikit-learn**: `train_test_split`, `fit`, `predict`, and MAE, RMSE and R2 on rows the model had not seen.
- **Encoding**: label encoding with an order for `Options` and `Gear_Type`, one-hot encoding for `Origin`, `Fuel_Type` and `Make`, and you measured what each was worth.
- **Scaling and transforming**: scaling made the weights comparable; predicting the log of the price cut the error in half again, from 40,000 to 20,000 SAR.

---

Made By **Sattam Altwaim**